# Elaborative Rehearsal (B) — Data Prep (**stage 2**)

`06`/`07` train B on XSum `(document -> one-sentence summary)`. That is a
**one-shot** compression objective, and B's inference loop is not one-shot —
`rehearse_elaborative` feeds the model its own prior output, chunk after
chunk. This notebook builds the rolling-shaped data that closes the gap.

## Why this cannot be skipped

C-DIC (arXiv:2606.12411) Table 1 measures exactly the shortcut:

| | PPL (MSC) |
|---|---|
| ICAE, trained one-shot, **applied incrementally** | **513.774** |
| ICAE, trained one-shot, applied one-shot | 27.656 |
| C-DIC | 8.431 |

Appendix F attributes the ~19x blow-up to the structural cause rather than to
tuning: an objective that never trained the compressor to consume and update
its *own* output drifts and compounds error when made to do so. Driving the
`07` checkpoint through B's loop is that same mismatch, so the failure is a
predicted outcome, not a risk.

## Why a linear running summary isn't enough either

`rehearse_elaborative` does not maintain one summary: it keeps a `ThreadMemory`
of separate topic threads, retrieves the ones related to the current chunk
(C-DIC Eq. 3), generates a new gist for *that chunk's thread* conditioned on
them, and the result *replaces* the matched slot (Eq. 6) rather than being
appended to a document-wide summary. A linear running-summary target would
only ever have one prior state to condition on, never a genuine choice among
several retrieved threads — a second, shape-level version of the same
train/inference mismatch C-DIC Table 1 measures.

This notebook curates the **threaded** shape directly: for each chunk,
retrieve related earlier thread(s) from a real `ThreadMemory`, ask the teacher
to shorten the chunk conditioned on them, and write the result back. See
`curation.py`'s comment above `TEACHER_SYSTEM_PROMPT_THREADED` for the full
argument.

## What this notebook produces

1. **Teacher-curated gists** — `curate_document_threaded` drives the same
   `ThreadMemory` class `rehearse_elaborative` uses at inference, with the
   teacher standing in for B at the generation step. Each gist is
   `format_elaborative_input(chunk, retrieved_context) -> gist`, the literal
   shape of one inference step.
2. **The "shorten", not "summarize" framing** — ReadAgent (Lee, Chen, Furuta,
   Canny & Fischer, ICML 2024, arXiv:2402.09727) §3.1: "shorten" preserves
   narrative flow across concatenation where "summarize" produces a
   restructured synopsis — exactly the failure mode here, since retrieved
   gists get concatenated with new chunks constantly.
3. **Self-conditioned pairs** — a fraction of training inputs carry what the
   *stage-1 model's own* `rehearse_elaborative` rollout actually retrieved,
   while the target stays the teacher's gist. This is the analogue of
   ra-TBPTT's purpose (removing exposure bias), **not** of its mechanism:
   C-DIC backpropagates one hop into a latent slot, and text slots are
   discrete, so no gradient can flow. Reporting it as an implementation of
   ra-TBPTT would be false.
4. **The retention curve** — C-DIC Fig. 2(a) with answer retention standing in
   for perplexity, measured against the full memory state (every slot), not
   just the chunk currently being touched.

> [!warning] This notebook spends API budget — 3 calls per chunk
> One teacher call plus two embedding calls (query + store) per chunk, and
> chunk *i*'s retrieval depends on chunk *i-1*'s write-back, so a document
> cannot be parallelised internally — documents can. §3 prints the call count
> and refuses to start until `CONFIRM_SPEND = True`. Results are cached per
> document, so an interrupted run resumes.
>
> The NIM endpoint has a real per-key rate limit; §4's `RateLimiter` and
> `MAX_WORKERS` are tuned to stay under it (see `configs/curation.yaml`'s
> `requests_per_second` comment). Budget for a nontrivial failure rate
> regardless — check §4's printed failure count before trusting §5.</cell id="6e8bd155">


In [2]:
import json
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import datasets
import pandas as pd
import yaml
from nltk.tokenize import sent_tokenize
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from src.pipeline.curation import (
    build_stage2_pairs_threaded,
    probe_accuracy_by_position,
    retention_probes,
)
from src.pipeline.embeddings import embed_texts, load_config as load_embed_config
from src.pipeline.rate_limit import RateLimiter
from src.pipeline.rehearsal import rehearse_elaborative
from src.pipeline.teacher import curate_document_threaded, load_curation_config
from src.pipeline.types import Chunk

CFG = load_curation_config()
EMBED_CFG = load_embed_config()
CHUNK_MAX_WORDS = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))["max_words"]
OUT_DIR = Path("data/processed/rehearsal_elaborative_stage2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"teacher          : {CFG['model']} (temperature {CFG['temperature']}, timeout {CFG['timeout']}s)")
print(f"embedding        : {EMBED_CFG['model']}")
print(f"chunk max_words  : {CHUNK_MAX_WORDS}")
print(f"document shape   : {CFG['doc_chunks']} chunks x {CFG['docs_per_corpus']} docs/corpus")
print(f"genres           : {CFG['genres']}")

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


teacher          : meta/llama-3.1-70b-instruct (temperature 0.0, timeout 120.0s)
embedding        : nvidia/llama-nemotron-embed-1b-v2
chunk max_words  : 200
document shape   : 12 chunks x 6 docs/corpus
genres           : ['news', 'narrativeqa', 'caselaw', 'wiki']


## 1. Corpora and the split

Corpus-level split, matching `04b`: chunks from one corpus share articles,
entities and phrasing, so a random chunk split would put near neighbours on
both sides and inflate validation scores.

**Four genres: news, narrativeqa, caselaw, wiki** — the same four `03`'s
baseline evaluates, so the generalizability claim isn't cherry-picked to the
genres most likely to show improvement. See `configs/curation.yaml`'s comment
for the per-genre reasoning: narrativeqa has the largest context-underuse gap
in the baseline (full_context 0.494 vs chunked_sequential 0.063) and its
single-continuous-narrative structure directly tests thread-memory tracking;
caselaw needed synthetic verbatim QA (`generate_caselaw_extractive_probes.py`)
since CaseHOLD's native questions have no answer span; wiki's full_context
ceiling (0.81) is already the highest of any genre, so a flat result there is
expected and still informative, not evidence of a bug.</cell id="c4778648">


In [3]:
GENRES = CFG["genres"]
N_TRAIN, N_VAL, N_TEST = CFG["n_train_corpora"], CFG["n_val_corpora"], CFG["n_test_corpora"]

splits = {"train": [], "val": [], "test": []}
for genre in GENRES:
    dirs = sorted((root / "data" / "processed" / f"{genre}_train").glob("corpus_*"))
    if not dirs:
        raise SystemExit(f"no corpora for {genre} — run data/scripts/build_{genre}_train_corpora.py")
    splits["train"] += dirs[:N_TRAIN]
    splits["val"] += dirs[N_TRAIN:N_TRAIN + N_VAL]
    splits["test"] += dirs[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_TEST]
    print(f"{genre}: {len(dirs)} corpora")

for name, dirs in splits.items():
    print(f"{name:5}: {len(dirs)} corpora")

news: 20 corpora
narrativeqa: 20 corpora
caselaw: 20 corpora
wiki: 20 corpora
train: 56 corpora
val  : 12 corpora
test : 12 corpora


## 2. Pseudo-documents

The corpora are ~50k words, i.e. ~250 chunks each — curating them whole would
be ~15,000 sequential calls per genre. Each corpus is instead cut into
**pseudo-documents** of `doc_chunks` consecutive chunks, sampled evenly across
the corpus so different articles are represented.

`doc_chunks = 12` is set by what the diagnostic needs, not by cost. C-DIC
Fig. 2(a) puts the static-compressor knee at 3-4 accumulated compressions, so a
document has to run well past it to show whether the curve keeps falling or
flattens out. 12 gives 8 positions beyond the knee.

Chunks are **fixed word windows**, not `paginate_semantic`'s embedding-based
boundaries — the same deliberate mismatch with inference that `04b` documents,
for the same reason (semantic chunking here would cost thousands of extra
embedding calls, and boundary choice affects which sentences sit together far
less than it affects the rolling objective this notebook is about).

Questions are attached to the chunk their `evidence_char_pos` falls inside, so
§5 can ask whether a fact read at chunk *j* is still in the summary *k*
revisions later.

In [4]:
def chunk_with_spans(text: str, max_words: int) -> list[dict]:
    """Consecutive `max_words`-word windows with their absolute char spans."""
    starts, cursor = [], 0
    for word in text.split():
        i = text.find(word, cursor)
        starts.append(i)
        cursor = i + len(word)
    return [
        {
            "text": text[starts[k]:(starts[k + max_words] if k + max_words < len(starts) else len(text))].strip(),
            "char_start": starts[k],
            "char_end": starts[k + max_words] if k + max_words < len(starts) else len(text),
        }
        for k in range(0, len(starts), max_words)
    ]


def build_documents(corpus_dir: Path) -> list[dict]:
    """`docs_per_corpus` documents of `doc_chunks` consecutive chunks, evenly
    spaced, each carrying the answer spans whose evidence lands in it."""
    text = (corpus_dir / "corpus.txt").read_text(encoding="utf-8")
    chunks = chunk_with_spans(text, CHUNK_MAX_WORDS)

    # caselaw's native questions.csv is CaseHOLD multiple-choice (no 'answer'
    # column at all) — questions_extractive.csv, if present, is this
    # project's own synthetic verbatim-answer QA over the same (HOLDING-token
    # -cleaned) corpus text, generated by
    # data/scripts/generate_caselaw_extractive_probes.py specifically so §5's
    # retention probe has something to check. news/narrativeqa never have
    # this file, so they fall through to their native questions.csv exactly
    # as before.
    extractive_path = corpus_dir / "questions_extractive.csv"
    questions_path = extractive_path if extractive_path.exists() else corpus_dir / "questions.csv"
    questions = pd.read_csv(questions_path)
    questions = questions.dropna(subset=["answer", "evidence_char_pos"]) if "answer" in questions.columns else questions.iloc[0:0]

    doc_chunks, n_docs = CFG["doc_chunks"], CFG["docs_per_corpus"]
    if len(chunks) < doc_chunks:
        return []
    # Evenly spaced, non-overlapping starts across the whole corpus.
    stride = max(doc_chunks, (len(chunks) - doc_chunks) // max(1, n_docs - 1)) if n_docs > 1 else doc_chunks
    starts = [s for s in range(0, len(chunks) - doc_chunks + 1, stride)][:n_docs]

    documents = []
    for doc_index, start in enumerate(starts):
        window = chunks[start:start + doc_chunks]
        answers_by_chunk = {}
        for position, chunk in enumerate(window):
            hits = questions[
                (questions["evidence_char_pos"] >= chunk["char_start"])
                & (questions["evidence_char_pos"] < chunk["char_end"])
            ] if len(questions) else questions
            answers_by_chunk[position] = [str(a) for a in hits["answer"].tolist()] if len(questions) else []
        documents.append({
            "key": f"{corpus_dir.parent.name}/{corpus_dir.name}/doc{doc_index:02d}",
            "genre": corpus_dir.parent.name.replace("_train", ""),
            "chunk_texts": [c["text"] for c in window],
            "answers_by_chunk": answers_by_chunk,
        })
    return documents


documents = {name: [d for c in dirs for d in build_documents(c)] for name, dirs in splits.items()}
for name, docs in documents.items():
    n_answers = sum(len(a) for d in docs for a in d["answers_by_chunk"].values())
    print(f"{name:5}: {len(docs):4} documents, {len(docs) * CFG['doc_chunks']:5} chunks, {n_answers:5} answer spans")

train:  336 documents,  4032 chunks,  9780 answer spans
val  :   72 documents,   864 chunks,  1971 answer spans
test :   72 documents,   864 chunks,  1916 answer spans


## 3. Cost gate

Three calls per chunk: one embedding to query the memory, one teacher call
to generate the gist, one embedding to store it. `PILOT_LIMIT` caps how many
*new* documents this run curates, so cost stays checkable before committing
to a large batch — run a small batch, check §5's retention `drop` on it, and
only then raise `PILOT_LIMIT` (or set it to `None`) to spend the rest.

The reason a limit matters at all: §5 can only check the documents that are
actually in `curation_cache.jsonl`. Curating everything in one pass before
ever looking at §5 leaves no budget to save if the teacher's own loop turns
out to collapse.

Set `CONFIRM_SPEND = True` only after reading the call count below. Curation
is cached per document in `curation_cache.jsonl`, so re-running skips what is
already done and an interrupted run resumes where it stopped.</cell id="e24792b5">


In [5]:
CACHE_PATH = OUT_DIR / "curation_cache.jsonl"
CONFIRM_SPEND = True  # <- flip to True to actually call the teacher
PILOT_LIMIT = 48  # <- curate at most this many NEW documents this run; None = no cap (spend the rest)

cached = {}
if CACHE_PATH.exists():
    for line in CACHE_PATH.read_text(encoding="utf-8").splitlines():
        if line.strip():
            record = json.loads(line)
            cached[record["key"]] = record


def interleave_by_genre(docs: list[dict]) -> list[dict]:
    """Round-robins across genres so any prefix of the result samples every
    genre roughly evenly, instead of exhausting one genre's whole quota
    before the next genre is even touched."""
    from itertools import zip_longest

    by_genre: dict[str, list[dict]] = {}
    for d in docs:
        by_genre.setdefault(d["genre"], []).append(d)
    return [d for group in zip_longest(*by_genre.values()) for d in group if d is not None]


# Train is still strictly prioritized first (it's what §5/§7 need most), but
# once train is exhausted, val and test are interleaved *with each other*
# too — not val-then-test in sequence. val/test are the same size (72 each),
# so a plain train->val->test concatenation would let a limited PILOT_LIMIT
# drain val completely before test is ever touched. zip_longest alternates
# one genre-interleaved val document with one genre-interleaved test
# document, so any prefix samples both splits (and all 4 genres) evenly.
from itertools import zip_longest

train_remaining = [d for d in interleave_by_genre(documents["train"]) if d["key"] not in cached]
val_interleaved = interleave_by_genre(documents["val"])
test_interleaved = interleave_by_genre(documents["test"])
val_test_remaining = [
    (split_name, d)
    for (val_d, test_d) in zip_longest(val_interleaved, test_interleaved)
    for split_name, d in (("val", val_d), ("test", test_d))
    if d is not None and d["key"] not in cached
]
todo_all = [("train", d) for d in train_remaining] + val_test_remaining
todo = todo_all[:PILOT_LIMIT] if PILOT_LIMIT is not None else todo_all

from collections import Counter

print(f"cached        : {len(cached)} documents")
print(f"remaining     : {len(todo_all)} documents")
print(f"this run      : {len(todo)} documents = {len(todo) * CFG['doc_chunks']} teacher calls "
      f"+ {2 * len(todo) * CFG['doc_chunks']} embedding calls")
print(f"this run by split+genre: {dict(Counter((split_name, d['genre']) for split_name, d in todo))}")
print(f"\nCONFIRM_SPEND = {CONFIRM_SPEND}   PILOT_LIMIT = {PILOT_LIMIT}")

todo = [d for _, d in todo]  # curate_document_threaded and §5-8 only need the document dict

if todo and not CONFIRM_SPEND:
    print("-> set CONFIRM_SPEND = True and re-run this cell's successor to start")
elif PILOT_LIMIT is not None and len(todo_all) > len(todo):
    print(f"-> pilot only. Check §5's retention drop on it, then raise PILOT_LIMIT and re-run §3-4 for the remaining {len(todo_all) - len(todo)} documents")

cached        : 336 documents
remaining     : 144 documents
this run      : 48 documents = 576 teacher calls + 1152 embedding calls
this run by split+genre: {('val', 'news'): 6, ('test', 'news'): 6, ('val', 'narrativeqa'): 6, ('test', 'narrativeqa'): 6, ('val', 'caselaw'): 6, ('test', 'caselaw'): 6, ('val', 'wiki'): 6, ('test', 'wiki'): 6}

CONFIRM_SPEND = True   PILOT_LIMIT = 48
-> pilot only. Check §5's retention drop on it, then raise PILOT_LIMIT and re-run §3-4 for the remaining 96 documents


## 4. Curate

Runs C-DIC's actual retrieve / generate / write-back loop
(`curate_document_threaded`) over one document at a time: for each chunk,
retrieve related earlier thread(s) by embedding similarity, ask the teacher to
shorten the chunk *conditioned on* what was retrieved, then write the result
back — replacing the matched thread if it was on-topic, opening a new one if
not. This is the literal shape `rehearse_elaborative` uses at inference.

Sequential *within* a document — chunk *i*'s retrieval depends on chunk
*i-1*'s write-back — but documents don't depend on each other, so documents
run concurrently via a thread pool. Two separate knobs control this:
`MAX_WORKERS` is how many documents are in flight at once (for overlapping
network wait); `rate_limiter` (`RateLimiter`, shared by every worker thread)
caps how fast requests actually get *issued*, since NIM enforces a per-key
rate limit independent of how many documents happen to be running. Raising
`MAX_WORKERS` doesn't by itself risk 429s — the shared limiter caps the
combined issue rate regardless. Tune `configs/curation.yaml`'s
`requests_per_second` from the printed failure count rather than guessing at
`MAX_WORKERS`.

All writes to `CACHE_PATH` and all mutation of `cached` happen back on the
main thread as each document's future completes — worker threads only ever
run the network calls inside `curate_document_threaded`. That sidesteps
locking `cache_file`/`cached` entirely rather than adding locks around them.
A crash costs at most `MAX_WORKERS` in-flight documents — finished documents
are durably cached, so a re-run resumes from there.

A failed teacher call falls back to the raw chunk text so positions stay
aligned; §7 drops pairs built from an unreliable position (its own call, or
its own retrieval, failed) rather than training on a silently degraded
target.</cell id="533f3c62">


In [6]:
if not CONFIRM_SPEND:
    print("skipped — CONFIRM_SPEND is False")
else:
    import threading
    from concurrent.futures import ThreadPoolExecutor, as_completed

    from tqdm.auto import tqdm

    MAX_WORKERS = 4  # <- documents in flight at once, for overlapping network wait.
    # Separate from request rate (rate_limiter below) — raising this doesn't
    # directly risk 429s, since every request from every worker draws from one
    # shared budget regardless of how many documents are concurrently in flight.

    # Shared across every worker thread — the actual fix for NIM's 429 rate
    # limit, which backoff alone can't prevent (backoff only reacts after a
    # request is already rejected). Tune configs/curation.yaml's
    # requests_per_second from this run's failure count.
    rate_limiter = RateLimiter(CFG["requests_per_second"])

    # Two progress bars: one document per completed future (outer), one chunk
    # per on_chunk_done callback pooled across every in-flight document
    # (inner) — with several documents running at once, per-document nested
    # bars would just stack up messily.
    progress_lock = threading.Lock()
    total_chunks = sum(len(document["chunk_texts"]) for document in todo)
    chunk_bar = tqdm(total=total_chunks, desc="chunks (all in-flight documents)")

    def _on_chunk_done(position, total):
        with progress_lock:
            chunk_bar.update(1)

    def _curate(document):
        result = curate_document_threaded(
            document["chunk_texts"], EMBED_CFG, CFG, embed_fn=embed_texts,
            on_chunk_done=_on_chunk_done, rate_limiter=rate_limiter,
        )
        return document, result

    # Worker threads only ever run curate_document_threaded's network calls.
    # Every write to CACHE_PATH and every mutation of `cached` happens here,
    # back on the main thread, as each future completes — that avoids needing
    # a lock around the file/dict instead of adding one.
    with CACHE_PATH.open("a", encoding="utf-8") as cache_file:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = [pool.submit(_curate, document) for document in todo]
            for future in tqdm(as_completed(futures), total=len(futures), desc="documents"):
                document, result = future.result()
                record = {
                    "key": document["key"],
                    "genre": document["genre"],
                    "chunk_texts": document["chunk_texts"],
                    "answers_by_chunk": {str(k): v for k, v in document["answers_by_chunk"].items()},
                    "gists": result.gists,
                    "context_texts": result.context_texts,
                    "write_back_actions": result.write_back_actions,
                    "memory_states": result.memory_states,
                    "failed_positions": result.failed_positions,
                    "retrieval_failed_positions": result.retrieval_failed_positions,
                    "store_failed_positions": result.store_failed_positions,
                }
                cache_file.write(json.dumps(record, ensure_ascii=False) + "\n")                                                                                                     
                cache_file.flush()
                cached[document["key"]] = record
    chunk_bar.close()

    failures = sum(len(r["failed_positions"]) for r in cached.values())
    embed_failures = sum(
        len(r["retrieval_failed_positions"]) + len(r["store_failed_positions"]) for r in cached.values()
    )
    print(f"\ncurated {len(cached)} documents, {failures} failed teacher calls, {embed_failures} failed embedding calls")

chunks (all in-flight documents):  73%|███████▎  | 420/576 [2:44:13<56:56, 21.90s/it]  [embeddings] Embedding API call failed (retries exhausted: 2, 1 texts, input_type='query'): ReadTimeout: HTTPSConnectionPool(host='integrate.api.nvidia.com', port=443): Read timed out. (read timeout=30.0)
[embeddings] Embedding API call failed (retries exhausted: 2, 1 texts, input_type='query'): ReadTimeout: HTTPSConnectionPool(host='integrate.api.nvidia.com', port=443): Read timed out. (read timeout=30.0)
[embeddings] Embedding API call failed (retries exhausted: 2, 1 texts, input_type='query'): ReadTimeout: HTTPSConnectionPool(host='integrate.api.nvidia.com', port=443): Read timed out. (read timeout=30.0)
chunks (all in-flight documents):  73%|███████▎  | 422/576 [2:46:08<1:35:56, 37.38s/it][embeddings] Embedding API call failed (retries exhausted: 2, 1 texts, input_type='query'): ReadTimeout: HTTPSConnectionPool(host='integrate.api.nvidia.com', port=443): Read timed out. (read timeout=30.0)
[embed


curated 384 documents, 306 failed teacher calls, 75 failed embedding calls


## 5. Does the teacher's own loop hold up?

Before training on these targets, check they are worth training on. This is
C-DIC Fig. 2(a) — perplexity against accumulated compressions — with **answer
retention** standing in for perplexity: a fact whose evidence was in chunk *j*,
is it still recoverable from the summary *k* revisions later?

The axis is **age**, not position, and that is not cosmetic. Probing every live
answer at each position is confounded: the number of live answers grows with
position while the summary stays a fixed-length summary, so accuracy would fall
with position even for a compressor that never degrades, and a downward curve
would be guaranteed in advance. Keyed by age, the probe count at each age is a
property of the corpus rather than of the axis.

Read `drop` (early minus late): clearly positive is the collapse signature. If
the *teacher* collapses here, the targets encode collapse and stage 2 would
train B to reproduce it — fix the prompt before proceeding.

In [7]:
train_records = [cached[d["key"]] for d in documents["train"] if d["key"] in cached]

if not train_records:
    print("no cached train documents yet — this reads as drop=0.000 with an empty")
    print("curve below, which is 'nothing to measure', NOT 'the teacher didn't degrade'.")
    print("Run §3-4 with CONFIRM_SPEND = True first.")

probes = []
for record in train_records:
    answers_by_chunk = {int(k): v for k, v in record["answers_by_chunk"].items()}
    # memory_states[i] is every slot's text concatenated after chunk i's
    # write-back — what a downstream reader would actually see at that point.
    # gists[i] alone is the wrong thing to probe here: a fact from chunk 2
    # might live in a slot chunk 9 never touches, and gists[9] would show
    # nothing about it even though the memory as a whole still carries it.
    probes += retention_probes(record["memory_states"], answers_by_chunk, CFG["probe_f1_threshold"])

# retention_probes returns one ProbeResult per age per document; pool them.
report = probe_accuracy_by_position(probes, collapse_after=3)
failure_rate = sum(len(r["failed_positions"]) for r in train_records) / max(
    1, sum(len(r["chunk_texts"]) for r in train_records)
)
print(f"train documents curated : {len(train_records)}")
print(f"teacher call failure rate: {failure_rate:.1%}   <- read this before trusting drop below")
print(f"probes pooled           : {sum(p.n_probes for p in probes)}")
print(f"overall retention : {report['overall_accuracy']:.3f}")
print(f"early (age < 3)   : {report['early_accuracy']:.3f}")
print(f"late  (age >= 3)  : {report['late_accuracy']:.3f}")

drop = report["drop"]
if not probes:
    verdict = "no probes pooled — see the note above, this is not a real reading"
elif failure_rate > 0.1:
    verdict = f"failure rate {failure_rate:.0%} is too high to trust this reading — fix reliability first, not the prompt"
elif drop > 0.05:
    verdict = "collapse signature — fix the prompt before spending the rest of the budget"
elif drop < -0.05:
    verdict = "retention improves with age (unexpected but not itself a collapse)"
else:
    verdict = "flat — no collapse detected"
print(f"drop              : {drop:+.3f}   <- {verdict}")

curve = pd.DataFrame(
    [{"age": age, "retention": value} for age, value in report["by_position"].items()]
)
display(curve)
curve.to_csv(OUT_DIR / "teacher_retention_by_age.csv", index=False)

train documents curated : 336
teacher call failure rate: 7.1%   <- read this before trusting drop below
probes pooled           : 64151
overall retention : 0.426
early (age < 3)   : 0.404
late  (age >= 3)  : 0.435
drop              : -0.031   <- flat — no collapse detected


,age,retention
0,0,0.407406
1,1,0.403604
2,2,0.400770
3,3,0.395548
4,4,0.402589
5,5,0.408491
6,6,0.414855
7,7,0.411534
8,8,0.429465
9,9,0.459274


## 6. Self-context from the stage-1 checkpoint

The context the model *actually* retrieves for itself, to be paired with the
gist the teacher says it should produce — the DAgger mix in §7.

Runs the stage-1 checkpoint through the real `rehearse_elaborative` loop
(`record_context_texts=True`) rather than a bespoke retrieval-free function:
self-conditioning has to reflect what *this* model's own `ThreadMemory`
retrieves, which depends on its own drifting gist embeddings, not on a
simplified one-slot-back approximation. This costs the same two embedding
calls per chunk `rehearse_elaborative` always costs — no new mechanism, just
reusing the inference-time function on the same chunks the teacher curated.

In [8]:
STAGE1_DIR = Path("experiments/rehearsal_elaborative_small")
SELF_PATH = OUT_DIR / "self_context_texts.json"

self_context_texts = json.loads(SELF_PATH.read_text(encoding="utf-8")) if SELF_PATH.exists() else {}

# Incremental, keyed by document — not "file exists -> trust it wholesale".
# Checking which keys are actually missing makes this self-healing if `cached`
# ever grows without this file being regenerated.
missing_keys = [k for k in cached if k not in self_context_texts]

if not missing_keys:
    print(f"self-context up to date for all {len(self_context_texts)} cached documents")
elif not STAGE1_DIR.exists():
    print(f"no stage-1 checkpoint at {STAGE1_DIR} — run 07 first.")
    print(f"Proceeding with self_conditioned_ratio = 0.0 (pure teacher forcing) for {len(missing_keys)} documents.")
else:
    from tqdm.auto import tqdm

    stage1_tokenizer = AutoTokenizer.from_pretrained(STAGE1_DIR)
    stage1_model = AutoModelForSeq2SeqLM.from_pretrained(STAGE1_DIR)

    for key in tqdm(missing_keys, desc="self-context"):
        record = cached[key]
        chunks = [Chunk(text=t, index=i) for i, t in enumerate(record["chunk_texts"])]
        _, self_records = rehearse_elaborative(
            chunks, stage1_model, stage1_tokenizer, EMBED_CFG, embed_texts,
            record_context_texts=True,
        )
        self_context_texts[key] = [r.retrieved_context_texts for r in self_records]
        SELF_PATH.write_text(json.dumps(self_context_texts, ensure_ascii=False), encoding="utf-8")
    print(f"generated self-context for {len(missing_keys)} new documents ({len(self_context_texts)} total)")

# Recompute coverage *after* the fill-in loop above — checking the
# pre-loop `missing_keys` snapshot here would report 0.0 even on a run that
# just finished computing every one of them.
still_missing = [k for k in cached if k not in self_context_texts]
RATIO = CFG["self_conditioned_ratio"] if not still_missing else 0.0
print(f"self_conditioned_ratio = {RATIO}" + (" (0.0: self-context coverage incomplete)" if still_missing else ""))

self-context: 100%|██████████| 48/48 [12:37<00:00, 15.78s/it]

generated self-context for 48 new documents (384 total)
self_conditioned_ratio = 0.3


## 7. Build the pairs

`build_stage2_pairs_threaded` needs no re-derivation: `input_text =
format_elaborative_input(chunk_texts[i], context_texts[i])`, `target_text =
gists[i]` — exactly the alignment `curate_document_threaded` already produced,
which is exactly the shape `rehearse_elaborative` feeds and expects back at
inference.

Pairs at a position whose own teacher call or own retrieval failed are
dropped — a target that fell back to raw chunk text, or a context built from a
failed retrieval, would teach degraded behaviour rather than the real
mechanism.

In [9]:
def pairs_for(record: dict) -> list[dict]:
    unreliable = set(record["failed_positions"]) | set(record["retrieval_failed_positions"])
    stage2 = build_stage2_pairs_threaded(
        record["chunk_texts"],
        record["context_texts"],
        record["gists"],
        self_context_texts=self_context_texts.get(record["key"]),
        self_conditioned_ratio=RATIO,
        seed=CFG["seed"],
    )
    return [
        {
            "key": record["key"],
            "genre": record["genre"],
            "input_text": p.input_text,
            "target_text": p.target_text,
            "chunk_position": p.chunk_position,
            "conditioning": p.conditioning,
        }
        for p in stage2
        if p.chunk_position not in unreliable
    ]


split_pairs = {}
for name, docs in documents.items():
    rows = [row for d in docs if d["key"] in cached for row in pairs_for(cached[d["key"]])]
    split_pairs[name] = rows
    if rows:
        frame = pd.DataFrame(rows)
        share = (frame["conditioning"] == "self").mean()
        print(f"{name:5}: {len(rows):5} pairs, {share:.1%} self-conditioned")
    else:
        print(f"{name:5}: 0 pairs")

if split_pairs["train"]:
    display(pd.DataFrame(split_pairs["train"])[["chunk_position", "conditioning", "input_text", "target_text"]].head(3))

train:  3742 pairs, 33.3% self-conditioned
val  :   270 pairs, 34.8% self-conditioned
test :   273 pairs, 34.4% self-conditioned


,chunk_position,conditioning,input_text,target_text
0,0,gold,current passage: (CNN) -- With his hands and ...,"Chester Arthur Stiles, 37, made his initial co..."
1,1,gold,current passage: the child on the videotape wa...,"Chester Arthur Stiles, 37, faces 23 felony cou..."
2,2,self,"current passage: Stiles' said that, before the...","Chester Arthur Stiles, 37, faces 23 felony cou..."


## 8. Tokenize and save

`INPUT_MAX_LENGTH` matches `06` at 512. The input is longer here than in stage
1 — it carries a chunk *plus* a prior summary — so the truncation rate is
measured rather than assumed. Right-truncation cuts the tail, and
`format_elaborative_input` puts the current passage first precisely so that
overflow costs the context and not the passage being rehearsed.

In [10]:
INPUT_MAX_LENGTH = 512
TARGET_MAX_LENGTH = 256  # a running summary over 12 chunks, not a one-sentence gist

tokenizer = AutoTokenizer.from_pretrained("t5-small")

if split_pairs["train"]:
    lengths = [len(tokenizer(r["input_text"], add_special_tokens=True).input_ids) for r in split_pairs["train"]]
    targets = [len(tokenizer(r["target_text"], add_special_tokens=True).input_ids) for r in split_pairs["train"]]
    over_input = sum(n > INPUT_MAX_LENGTH for n in lengths) / len(lengths)
    over_target = sum(n > TARGET_MAX_LENGTH for n in targets) / len(targets)
    print(f"input  tokens: mean {sum(lengths)/len(lengths):.0f}, max {max(lengths)}, over {INPUT_MAX_LENGTH}: {over_input:.1%}")
    print(f"target tokens: mean {sum(targets)/len(targets):.0f}, max {max(targets)}, over {TARGET_MAX_LENGTH}: {over_target:.1%}")


def tokenize(batch):
    encoded = tokenizer(batch["input_text"], max_length=INPUT_MAX_LENGTH, truncation=True)
    labels = tokenizer(text_target=batch["target_text"], max_length=TARGET_MAX_LENGTH, truncation=True)
    encoded["labels"] = labels["input_ids"]
    return encoded


if split_pairs["train"]:
    for name, rows in split_pairs.items():
        pd.DataFrame(rows).to_csv(OUT_DIR / f"{name}_pairs_raw.csv", index=False)

    dataset = datasets.DatasetDict({
        name: datasets.Dataset.from_pandas(pd.DataFrame(rows), preserve_index=False)
        for name, rows in split_pairs.items() if rows
    })
    tokenized = dataset.map(tokenize, batched=True, remove_columns=["input_text", "target_text"])
    tokenized.save_to_disk(str(OUT_DIR))
    print(f"\nsaved to {OUT_DIR}")
    print(tokenized)
else:
    print("nothing to save — run §4 with CONFIRM_SPEND = True first")

Token indices sequence length is longer than the specified maximum sequence length for this model (609 > 512). Running this sequence through the model will result in indexing errors


input  tokens: mean 497, max 2043, over 512: 34.3%
target tokens: mean 258, max 656, over 256: 40.5%


Saving the dataset (1/1 shards): 100%|██████████| 273/273 [00:00<00:00, 58373.01 examples/s]


saved to data/processed/rehearsal_elaborative_stage2
DatasetDict({
    train: Dataset({
        features: ['key', 'genre', 'chunk_position', 'conditioning', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3742
    })
    val: Dataset({
        features: ['key', 'genre', 'chunk_position', 'conditioning', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 270
    })
    test: Dataset({
        features: ['key', 'genre', 'chunk_position', 'conditioning', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 273
    })
})


## Summary

| | stage 1 (`06`/`07`) | stage 2 (this notebook / `07b`) |
|---|---|---|
| source | XSum articles | the project's own train corpora |
| objective | `document -> one-sentence summary` (one-shot) | `(retrieved context, C_i) -> gist_i` (rolling, threaded) |
| input template | bare document | `format_elaborative_input` — B's inference shape, exactly |
| conditioning | none | teacher's own retrieval, mixed with the model's own |

### This run

- **336 train documents curated** — 84 each across news, narrativeqa,
  caselaw, wiki (full train quota, evenly balanced).
- **Teacher call failure rate: 7.1%** (287/4032 chunks) — under the 10%
  trust threshold, so the retention reading below is trustworthy.
- **Retention: flat, no collapse.** drop = -0.031 (early 0.404, late 0.435) —
  if anything, retention rises at the oldest ages (0.49-0.54 at age 10-11
  vs ~0.40 early), the opposite of a collapse signature.
- **3742 training pairs**, 33.3% self-conditioned (target ~30%, on the DAgger
  mix's own terms). val/test are 0 pairs — this run only curated train.
- **Token lengths**: input mean 497 (34.3% over the 512 cap), target mean 258
  (40.5% over the 256 cap) — truncation is real but not dominant; right-
  truncation costs the retrieved context first, not the current passage.

### What has been imported from C-DIC and what has not

- **Eq. 3 retrieval + Eq. 4 conditioned compression** — `curate_document_threaded`
  drives a real `ThreadMemory`, so a gist is generated *with* its retrieved
  context in view, not independently of it.
- **Eq. 6 insert/revise** — executed by `ThreadMemory.write_back` during
  curation, identically to how it executes at inference.
- **Fig. 2(a) collapse curve** — §5, with retention for perplexity, re-keyed
  from position to age to remove the growing-probe-count confound, and probed
  against the full memory state rather than a single chunk's gist.
- **ra-TBPTT** — *not* implemented. §6/§7 mix in what the stage-1 model's own
  rollout actually retrieves, which addresses the exposure bias ra-TBPTT
  exists to remove but shares none of its mechanism; text slots are discrete
  and no gradient can reach them. Say so in the writeup.

Next: `07b_rehearsal_elaborative_stage2_train.ipynb`.